## 1. Model Architecture
- Diagram of the network (input → conv layers → dense → output)
- Layer-by-layer breakdown with dimensions
- Loss function and optimizer choice
- Hyperparameters table

---

In [1]:
import torch
import torch.nn as nn

class Layer_One_control(nn.Module):
    def __init__(self):
        super().__init__()
        # define your layers here
        self.conv1 = nn.Conv2d(12, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 8 * 8, 2048)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(2048, 4096)

    def forward(self, x):
        # define how data flows through the layers
        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.conv3(x)
        x = self.relu(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## 2. Thought Process

#### Why this architecture?

This is basically an initial model to test out a basic chess model using CNNs and dense layers. The starting point of this project.
The design idea behind a CNN is that it is good to detect spatial patterns, something essential in a chess game. The expected structure
we would like the CNN to follow is learning the spatial features of the board in the conv layers, which piece is where, who attacks whom 
and the later dense layers then decide what to do with that information, deciding on which move to make

#### Design decisions and tradeoffs

| Decision | Choice | Reasoning |
|---|---|---|
| Model type | CNN + Dense |Basic, easy to test|
| Kernel size | 3×3 |Each convolution sees the squares around as well|
| Conv layers | 3 (32→64→128 filters) |Increasing filters to detect more complex patterns at each level, 3 layers allows the model to see the whole board|
| Dense layers | 8192→2048→4096 |Decreasing number of neurons, 4096 being the expected output shape |
| Activation | ReLU |Standard for hidden layers |
| Dropout | 0.3 |To avoid overfitting |
| Optimizer | Adam (lr=0.001) |Safe default |
| Batch size | 512 |Optimal time/accuracy for my setup |
| Loss function | Cross-Entropy |Used for classification |

#### What we expect to work and what might not

**Should work:**
- Makes logical moves
- Has decent strategies
- Should have a solid elo (expecting 800-1200)


**Might struggle:**
- Performance ceiling is limited by the strength of the training data (2200+ Lichess) (~2000 chess.com)
- Each move is independent, so ideas could be forgotten in between moves




## 3.Training

We train the model using **Cross-Entropy Loss** and the **Adam optimizer** (lr=0.001). Data is loaded in batches of 64, shuffled each epoch for training.

| Hyperparameter | Value |
|---|---|
| Batch Size | 512 |
| Learning Rate | 0.001 |
| Optimizer | Adam |
| Loss Function | Cross-Entropy |
| Dropout | 0.3 |
| Epochs | 10 |

After each epoch, we evaluate on the validation set and track loss and accuracy.

#### Load & Prepare Data

Load the 50k dataset (train and val splits) from `.npy` files, convert to PyTorch tensors, and wrap in DataLoaders for batched iteration.

In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from tqdm import tqdm

X_train = np.load("../../data/formatted_data/50k/X-50k-train.npy")
y_train = np.load("../../data/formatted_data/50k/Y-50k-train.npy")
X_val = np.load("../../data/formatted_data/50k/X-50k-val.npy")
y_val = np.load("../../data/formatted_data/50k/Y-50k-val.npy")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)

#### Initialize Model, Loss & Optimizer

Instantiate the control model, define the loss function (Cross-Entropy) and optimizer (Adam, lr=0.001).

In [4]:
device = torch.device("mps")
model = Layer_One_control().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#### Training Loop

For each epoch: train on all batches, then evaluate on the validation set. Prints train loss, validation loss, and validation accuracy per epoch.

In [6]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    train_loss = running_loss / len(train_dataloader)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in val_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = model(batch_X)
            loss = criterion(output, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    val_loss /= len(val_dataloader)
    val_acc = correct / total
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1: 100%|██████████| 7670/7670 [04:16<00:00, 29.92it/s]


Epoch 1 | Train Loss: 3.4359 | Val Loss: 3.2068 | Val Acc: 0.2367


Epoch 2: 100%|██████████| 7670/7670 [04:15<00:00, 30.01it/s]


Epoch 2 | Train Loss: 3.1882 | Val Loss: 3.0947 | Val Acc: 0.2517


Epoch 3: 100%|██████████| 7670/7670 [04:14<00:00, 30.11it/s]


...

Epoch 8 | Train Loss: 2.6870 | Val Loss: 3.0139 | Val Acc: 0.2664


Epoch 9: 100%|██████████| 7670/7670 [04:17<00:00, 29.76it/s]


Epoch 9 | Train Loss: 2.6435 | Val Loss: 3.0237 | Val Acc: 0.2676


Epoch 10: 100%|██████████| 7670/7670 [04:18<00:00, 29.63it/s]


Epoch 10 | Train Loss: 2.6053 | Val Loss: 3.0543 | Val Acc: 0.2669


saving model

In [7]:
torch.save(model.state_dict(), "../../models/layer-one-50k.pth")

Methods import

In [8]:
import chess
# Piece type to plane index mapping
PIECE_TO_PLANE = {
    (chess.PAWN, chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK, chess.WHITE): 3,
    (chess.QUEEN, chess.WHITE): 4,
    (chess.KING, chess.WHITE): 5,
    (chess.PAWN, chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK, chess.BLACK): 9,
    (chess.QUEEN, chess.BLACK): 10,
    (chess.KING, chess.BLACK): 11,
}

def board_to_tensor(board):
    """
    Converts a python-chess Board object into a 12x8x8 numpy tensor.

    Each of the 12 planes represents one piece type and color:
        0: White Pawns     6: Black Pawns
        1: White Knights   7: Black Knights
        2: White Bishops   8: Black Bishops
        3: White Rooks     9: Black Rooks
        4: White Queen    10: Black Queen
        5: White King     11: Black King

    Each plane is an 8x8 grid where 1 indicates the piece is on that
    square and 0 indicates it is not.

    Args:
        board: a chess.Board object representing the current position.

    Returns:
        np.ndarray of shape (12, 8, 8) with binary values.
    """
    
    tensor = np.zeros((12, 8, 8), dtype=np.uint8)
    
    for (piece_type, color), plane in PIECE_TO_PLANE.items():
        for square in board.pieces(piece_type, color):
            row = square // 8
            col = square % 8
            tensor[plane][row][col] = 1
    return tensor



game structure

In [9]:
import chess.engine

def prediction_to_move(number):
    from_square = number // 64
    to_square = number % 64

    letters = 'abcdefgh'
    from_letter = letters[from_square % 8]
    from_number = from_square // 8 + 1

    to_letter = letters[to_square % 8]
    to_number = to_square // 8 + 1

    return f'{from_letter}{from_number}{to_letter}{to_number}'

def model_move(board):
    # Model's turn to move
    model.eval()
    with torch.no_grad():
        tensor = board_to_tensor(board)
        input = torch.tensor(tensor, dtype=torch.float32).unsqueeze(0).to(device)
        output = model(input)

    # Converting outputs to move format
    sorted_output = torch.argsort(output[0], descending=True)

    for i in sorted_output:
        move = prediction_to_move(i)
        if move[:2] == move[2:]:  # same square, skip
            continue
        # Promotion check
        from_sq = i // 64
        to_sq = i % 64
        piece = board.piece_at(from_sq)
        if piece and piece.piece_type == chess.PAWN and (to_sq // 8 == 7 or to_sq // 8 == 0):
            move += "q"

        move = chess.Move.from_uci(move)

        if move in board.legal_moves:
            board.push(move)
            break
    
def stockfish_move(board, engine):
    # Ask Stockfish for a move (with a time limit)
    
    result = engine.play(board, chess.engine.Limit(time=0.1))
    board.push(result.move)

            


def stockfish_game_sim(elo, model_color):
    board = chess.Board()
    engine = chess.engine.SimpleEngine.popen_uci("stockfish")

    # Set Stockfish to a specific ELO
    engine.configure({"UCI_LimitStrength": True, "UCI_Elo": elo})

    moves = 0
    
    if model_color == 'white':
        over = False
        
        while over == False:

            model_move(board)   
            moves += 1     
            if board.is_game_over():
                over = True
                engine.quit()

                if board.is_checkmate():
                    
                    return (1, "checkmate")
                
                else:
                    return (0, "draw")

            stockfish_move(board, engine)

            if board.is_game_over():
                over = True
                engine.quit()
     
                if board.is_checkmate():
                    return (-1, "checkmate")
                
                else:
                    return (0, "draw")

    elif model_color == 'black':
        over = False
        
        while over == False:

            stockfish_move(board, engine)        
            if board.is_game_over():
                over = True
                engine.quit()
   
                if board.is_checkmate():
                    return (-1, "checkmate")
                
                else:
                    return (0, "draw")

            model_move(board)

            if board.is_game_over():
                over = True
                engine.quit()
               
                if board.is_checkmate():
                    return (1, "checkmate")
                
                else:
                    return (0, "draw")

            
wins = 0
draws = {"stalemate": 0, "insufficient material": 0, "threefold repetition": 0, "fifty-move rule": 0, "draw": 0}
losses = 0

for i in tqdm(range(5)):
    result = stockfish_game_sim(1320, 'white')
    if result[0] == 1:
        wins += 1
    elif result[0] == 0:
        draws[result[1]] += 1
    elif result[0] == -1:
        losses += 1

print(f'white wins: {wins}')
print(f'white draws: {sum(draws.values())} — {draws}')
print(f'white losses: {losses}')

wins = 0
draws = {"stalemate": 0, "insufficient material": 0, "threefold repetition": 0, "fifty-move rule": 0, "draw": 0}
losses = 0

for i in tqdm(range(5)):
    result = stockfish_game_sim(1320, 'black')
    if result[0] == 1:
        wins += 1
    elif result[0] == 0:
        draws[result[1]] += 1
    elif result[0] == -1:
        losses += 1

print(f'black wins: {wins}')
print(f'black draws: {sum(draws.values())} — {draws}')
print(f'black losses: {losses}')
            

100%|██████████| 5/5 [00:23<00:00,  4.78s/it]


white wins: 0
white draws: 2 — {'stalemate': 0, 'insufficient material': 0, 'threefold repetition': 0, 'fifty-move rule': 0, 'draw': 2}
white losses: 3


100%|██████████| 5/5 [00:23<00:00,  4.74s/it]

black wins: 0
black draws: 0 — {'stalemate': 0, 'insufficient material': 0, 'threefold repetition': 0, 'fifty-move rule': 0, 'draw': 0}
black losses: 5


In [10]:
import random
from tqdm import tqdm

def random_move(board):
    move = random.choice(list(board.legal_moves))
    board.push(move)

def random_game_sim(model_color):
    board = chess.Board()

    moves = 0
    
    if model_color == 'white':
        over = False
        
        while over == False:

            model_move(board)   
            moves += 1     
            if board.is_game_over():
                over = True

                #print(moves)
                if board.is_checkmate():
                    
                    return (1, "checkmate")
                
                else:
                    if board.is_stalemate():
                        return (0, "stalemate")
                    elif board.is_insufficient_material():
                        return (0, "insufficient material")
                    elif board.can_claim_threefold_repetition():
                        return (0, "threefold repetition")
                    elif board.can_claim_fifty_moves():
                        return (0, "fifty-move rule")
                    else:
                        return (0, "draw")

            random_move(board)

            if board.is_game_over():
                over = True

                #print(moves)
                if board.is_checkmate():
                    return (-1, "checkmate")
                
                else:
                    if board.is_stalemate():
                        return (0, "stalemate")
                    elif board.is_insufficient_material():
                        return (0, "insufficient material")
                    elif board.can_claim_threefold_repetition():
                        return (0, "threefold repetition")
                    elif board.can_claim_fifty_moves():
                        return (0, "fifty-move rule")
                    else:
                        return (0, "draw")

    elif model_color == 'black':
        over = False
        
        while over == False:

            random_move(board)       
            if board.is_game_over():
                over = True
   
                #print(moves)
                if board.is_checkmate():
                    return (-1, "checkmate")
                
                else:
                    if board.is_stalemate():
                        return (0, "stalemate")
                    elif board.is_insufficient_material():
                        return (0, "insufficient material")
                    elif board.can_claim_threefold_repetition():
                        return (0, "threefold repetition")
                    elif board.can_claim_fifty_moves():
                        return (0, "fifty-move rule")
                    else:
                        return (0, "draw")

            model_move(board)

            if board.is_game_over():
                over = True
   
                #print(moves)
                if board.is_checkmate():
                    return (1, "checkmate")
                
                else:
                    if board.is_stalemate():
                        return (0, "stalemate")
                    elif board.is_insufficient_material():
                        return (0, "insufficient material")
                    elif board.can_claim_threefold_repetition():
                        return (0, "threefold repetition")
                    elif board.can_claim_fifty_moves():
                        return (0, "fifty-move rule")
                    else:
                        return (0, "draw")
                
wins = 0
draws = {"stalemate": 0, "insufficient material": 0, "threefold repetition": 0, "fifty-move rule": 0, "draw": 0}
losses = 0

for i in tqdm(range(200)):
    result = random_game_sim('white')
    if result[0] == 1:
        wins += 1
    elif result[0] == 0:
        draws[result[1]] += 1
    elif result[0] == -1:
        losses += 1

print(f'white wins: {wins}')
print(f'white draws: {sum(draws.values())} — {draws}')
print(f'white losses: {losses}')

wins = 0
draws = {"stalemate": 0, "insufficient material": 0, "threefold repetition": 0, "fifty-move rule": 0, "draw": 0}
losses = 0

for i in tqdm(range(200)):
    result = random_game_sim('black')
    if result[0] == 1:
        wins += 1
    elif result[0] == 0:
        draws[result[1]] += 1
    elif result[0] == -1:
        losses += 1

print(f'black wins: {wins}')
print(f'black draws: {sum(draws.values())} — {draws}')
print(f'black losses: {losses}')

100%|██████████| 200/200 [00:46<00:00,  4.30it/s]


white wins: 117
white draws: 83 — {'stalemate': 73, 'insufficient material': 0, 'threefold repetition': 10, 'fifty-move rule': 0, 'draw': 0}
white losses: 0


100%|██████████| 200/200 [00:51<00:00,  3.85it/s]

black wins: 91
black draws: 109 — {'stalemate': 89, 'insufficient material': 0, 'threefold repetition': 20, 'fifty-move rule': 0, 'draw': 0}
black losses: 0


chess.com bot testing

In [197]:
board = chess.Board()

In [258]:
model_move(board)
print(board.peek())

f8h6


In [257]:
board.push_uci("g8h8")

Move.from_uci('g8h8')

Vizualisation

In [259]:
from torchviz import make_dot

dummy = torch.randn(1, 12, 8, 8).to(device)
output = model(dummy)
dot = make_dot(output, params=dict(model.named_parameters()))
dot.render("layer_one_control", format="png")

'layer_one_control.png'

In [260]:
from torchinfo import summary
summary(model, input_size=(1, 12, 8, 8))

Layer (type:depth-idx)                   Output Shape              Param #
Layer_One_control                        [1, 4096]                 --
├─Conv2d: 1-1                            [1, 32, 8, 8]             3,488
├─ReLU: 1-2                              [1, 32, 8, 8]             --
├─Conv2d: 1-3                            [1, 64, 8, 8]             18,496
├─ReLU: 1-4                              [1, 64, 8, 8]             --
├─Conv2d: 1-5                            [1, 128, 8, 8]            73,856
├─ReLU: 1-6                              [1, 128, 8, 8]            --
├─Flatten: 1-7                           [1, 8192]                 --
├─Linear: 1-8                            [1, 2048]                 16,779,264
├─ReLU: 1-9                              [1, 2048]                 --
├─Dropout: 1-10                          [1, 2048]                 --
├─Linear: 1-11                           [1, 4096]                 8,392,704
Total params: 25,267,808
Trainable params: 25,267,808
Non-t

In [7]:
import torch.onnx



dummy = torch.randn(1, 12, 8, 8).to(device)
torch.onnx.export(model, dummy, "layer_one_control.onnx")

import netron
netron.start("layer_one_control.onnx")

/var/folders/27/fgn00k7j28bb5nnr4qprcw2c0000gn/T/ipykernel_76655/612002143.py:6: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(model, dummy, "layer_one_control.onnx")
W0411 10:56:43.057000 76655 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0411 10:56:43.058000 76655 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0411 10:56:43.059000 76655 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Layer_One_control([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Layer_One_control([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Serving 'layer_one_control.onnx' at http://localhost:8080


('localhost', 8080)

## 4. Results
- Training curves (loss, accuracy)
- Stockfish evaluation metrics
- Example games (good moves, bad moves, funny blunders)
- Final ELO estimate
- Key takeaways and what to improve in the next bot